# KadiPy - Module `kidas` : Documentation complète

> **kidas** (*Kadi Data Acquisition & Standardization*) est le module de manipulation de données du projet KadiPy.
> Il est conçu pour les besoins spécifiques de l'AgriTech en Afrique de l'Ouest (Bénin).

---

## Architecture générale

```
kadi.kidas
├── sources/
│   ├── base.py          → Source (classe abstraite)
│   ├── csv_source.py    → CSVSource
│   ├── excel_source.py  → ExcelSource
│   ├── json_source.py   → JSONSource
│   ├── netcdf_source.py → NetCDFSource
│   └── api_source.py    → APISource
├── cleaner.py           → Cleaner
├── validator.py         → Validator
├── normalizer.py        → Normalizer
├── cache.py             → Cache
└── pipeline.py          → Pipeline (point d'entrée)
```

## Table des notebooks à créer

| # | Notebook | Classe | Description |
|---|---|---|---|
| 01 | [01_csv_source.ipynb](./01_csv_source.ipynb) | `CSVSource` | CSV : encodage auto, délimiteur auto, fallbacks |
| 02 | [02_excel_source.ipynb](./02_excel_source.ipynb) | `ExcelSource` | Excel : cellules fusionnées, multi-feuilles, en-tête auto |
| 03 | [03_json_source.ipynb](./03_json_source.ipynb) | `JSONSource` | JSON : aplatissement imbriqué, dict ou fichier |
| 04 | [04_netcdf_source.ipynb](./04_netcdf_source.ipynb) | `NetCDFSource` | NetCDF : CHIRPS/ERA5, bbox Bénin, Dask |
| 05 | [05_api_source.ipynb](./05_api_source.ipynb) | `APISource` | API REST : backoff exponentiel, rate limiting |
| 06 | [06_data_cleaner.ipynb](./06_data_cleaner.ipynb) | `Cleaner` | Nettoyage : doublons, NaN, outliers, dates, texte |
| 07 | [07_data_validator.ipynb](./07_data_validator.ipynb) | `Validator` | Validation : schéma, plages, GPS, unicité, score qualité |
| 08 | [08_data_normalizer.ipynb](./08_data_normalizer.ipynb) | `Normalizer` | Normalisation : snake_case, FAO, marchés Bénin, unités |
| 09 | [09_data_cache.ipynb](./09_data_cache.ipynb) | `Cache` | Cache SQLite : save/load, historique, expiration |
| 10 | [10_data_pipeline.ipynb](./10_data_pipeline.ipynb) | `Pipeline` | Orchestrateur : API fluide, pipeline complet |

---

## Démarrage rapide

In [ ]:
# Installation des dépendances (si nécessaire)
# !pip install pandas openpyxl chardet scipy shapely Unidecode xarray requests

In [ ]:
import os
import pandas as pd

import kadi.kidas as kidas
print(f"kidas version : {kidas.__version__}")
print(f"Classes disponibles : {kidas.__all__}")

#### Explication de la sortie - Cellule 1 : Importation et vérification du module

Cette cellule vérifie que le module `kidas` est correctement installé et accessible depuis le chemin projet.

| Ligne affichée | Signification |
|---|---|
| `kidas version : 1.2.0` | La version courante du module `kidas` est **1.0.0**, définie dans `kidas.__version__`. |
| `Classes disponibles : [...]` | La liste `kidas.__all__` expose les **11 symboles publics** du module : les 5 sources de données (`CSVSource`, `ExcelSource`, `JSONSource`, `NetCDFSource`, `APISource`), les 4 composants de traitement (`Cleaner`, `Validator`, `Normalizer`, `Cache`), l'orchestrateur (`Pipeline`) et la fonction utilitaire (`load_clean`). |

> **En résumé :** Si cette cellule s'exécute sans erreur d'import, l'environnement est correctement configuré et tous les composants de `kidas` sont disponibles.

### Exemple minimal : charger un fichier CSV

In [ ]:
import os
import tempfile
import pandas as pd
import numpy as np

# Création d'un fichier de démonstration
df_test = pd.DataFrame({
    'culture': ['Maïs', 'Niébé', 'igname', 'MAÏS', None],
    'prix_xof': [350, 500, 250, 350, np.nan],
    'rendement_kg': [1500.0, 800.0, 2000.0, 1500.0, np.nan],
})

fichier = os.path.join(tempfile.gettempdir(), 'demo_index.csv')
df_test.to_csv(fichier, index=False)

# Chargement et nettoyage en une ligne
df_propre, rapport = kidas.load_clean(fichier, cache=False)

print(f"Avant : {len(df_test)} lignes | Après : {len(df_propre)} lignes")
df_propre

#### Explication de la sortie - Cellule 2 : Chargement CSV minimal (`load_clean`)

**Contexte :** Un DataFrame de démonstration est créé avec 5 lignes intentionnellement imparfaites
(doublons de casse, valeur `None`, `NaN`), sauvegardé en CSV temporaire, puis rechargé via `load_clean`.

**Ligne texte affichée :**
```
Avant : 5 lignes | Après : 5 lignes
```
Le nombre de lignes reste **5** car `load_clean` avec les paramètres par défaut applique
un nettoyage conservateur : les valeurs manquantes **numériques** sont imputées à la moyenne
de leur colonne, sans supprimer aucune ligne.

**Tableau affiché - analyse ligne par ligne :**

| index | culture | prix\_xof | rendement\_kg | Explication |
|---|---|---|---|---|
| 0 | `Maïs` | 350.0 | 1500.0 | Ligne originale, inchangée. |
| 1 | `Niébé` | 500.0 | 800.0 | Ligne originale, inchangée. |
| 2 | `igname` | 250.0 | 2000.0 | Casse originale conservée - pas de normalisation de texte à ce stade. |
| 3 | `MAÏS` | 350.0 | 1500.0 | Doublon de `Maïs` (casse différente) **conservé** - `drop_dupes` est sensible à la casse par défaut. |
| 4 | `NaN` | 362.5 | 1450.0 | `culture` est `None` → reste `NaN` (texte non imputable). `prix_xof` et `rendement_kg`, initialement `NaN`, sont **imputés à la moyenne** des 4 autres valeurs : (350+500+250+350)/4 = **362.5** et (1500+800+2000+1500)/4 = **1450.0**. |

> **À retenir :** `load_clean` est un raccourci pratique pour un nettoyage de base.
> Pour un contrôle fin (suppression de doublons par casse, normalisation du texte, mapping FAO…),
> utiliser le pipeline complet avec `Pipeline` comme montré dans l'exemple suivant.

### Exemple avancé : pipeline personnalisé complet

In [ ]:
import tempfile
import os
import pandas as pd

from kadi.kidas import Pipeline, Cache

cache_dir = os.path.join(tempfile.gettempdir(), 'kidas_index_cache')

pipeline = Pipeline()
pipeline._cache = Cache(cache_dir=cache_dir)

df_final, rapport = (
    pipeline
    .add_source(fichier)
    .add_step('drop_dupes')
    .add_step('fill_missing', strategy='mean')
    .add_step('norm_text', cols=['culture'], case='lower')
    .validate({'culture': 'str', 'prix_xof': 'float'})
    .add_step("std_crops", col="culture")
    .run(cache=False)
)

print(f"Score qualité : {rapport.get('quality_score', {}).get('overall', 'N/A')}")
print(f"Étapes appliquées : {rapport['steps_summary']}")
df_final

Le pipeline configure la source, transforme les données et affiche le score ainsi que `steps_summary`, la liste des étapes exécutées.

In [ ]:
# Réutilisation du pipeline
pipeline_propre = Pipeline()
pipeline_propre._cache = Cache(cache_dir=cache_dir)

df_final, rapport = (
    pipeline_propre
    .add_source(fichier)
    .add_step('drop_dupes')
    .add_step('fill_missing', strategy='mean')
    .add_step('norm_text', cols=['culture'], case='lower')
    .validate({'culture': 'str', 'prix_xof': 'float'})
    .add_step("std_crops", col="culture")
    .run(cache=False)
)

score = rapport.get('quality_score', {})
print(f"Score qualité global : {score.get('overall', 'N/A')}")
df_final

Le même enchaînement peut être relancé avec une nouvelle instance de `Pipeline`.

---

## Flux de données complet

```
Fichier CSV / Excel / JSON / NetCDF   API REST
          ↓                              ↓
     Source (lecture + métadonnées)
          ↓
     Cleaner (doublons, NaN, outliers, dates, texte)
          ↓
     Validator (schéma, plages, GPS, unicité → score qualité)
          ↓
     Normalizer (snake_case, FAO, marchés Bénin, unités)
          ↓
     Cache (SQLite, compression, versioning)
          ↓
     pd.DataFrame prêt à l'emploi + rapport complet
```

## Exceptions

Toutes les exceptions sont centralisées dans `kadi/exceptions.py` :

| Exception | Levée par |
|---|---|
| `ConnectError` | Fichier/API inaccessible |
| `ReadError` | Erreur de lecture/décodage |
| `KidasWriteError` | Erreur d'écriture |
| `KidasCleaningError` | Méthode ou paramètre invalide |
| `KidasValidationError` | Données invalides |
| `KidasCacheError` | Problème de base SQLite |
| `KidasPipelineError` | Pipeline mal configuré |